In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

import torchvision
# For image transforms
from torchvision import transforms
# For DATA SET
import torchvision.datasets as datasets
# For Pytorch methods
import torch
import torch.nn as nn
# For Optimizer
import torch.optim as optim
# FOR DATA LOADER
from torch.utils.data import DataLoader

from denoising_diffusion_pytorch import Unet, GaussianDiffusion

In [ ]:
LEARNING_RATE = 4e-4
BATCH_SIZE = 128
N_EPOCHS = 100

IMAGE_SIZE = 28
TIME_STEPS = 1000
SAMPLING_TIMESTEPS = 250

DIM = 32
DIM_MULTS = (1, 2, 5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# for testing
# N_EPOCHS = 1
# BATCH_SIZE = 64
# SAMPLING_TIMESTEPS = 50

In [ ]:
# Load mnist

myTransforms = transforms.Compose([
    transforms.ToTensor()
])

print("Loading MNIST digits dataset")

dataset = datasets.MNIST(
    root="../dataset/",
    train=True,
    transform=myTransforms,
    download=True
)

test_dataset = datasets.MNIST(
    root="../dataset/",
    train=False,
    transform=myTransforms,
    download=True
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training samples:", len(dataset))
print("Test samples:", len(test_dataset))

In [ ]:
# Visualize real mnist digits

real_images, real_labels = next(iter(loader))

grid = torchvision.utils.make_grid(real_images[:16], nrow=4)

plt.figure(figsize=(5, 5))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.axis("off")
plt.title("Real MNIST training images")
plt.show()

In [ ]:
# U-net
model = Unet(
    dim=DIM,
    dim_mults=DIM_MULTS,
    flash_attn=False,
    channels=1
)

# Diffusion model
diffusion = GaussianDiffusion(
    model,
    image_size=IMAGE_SIZE,
    timesteps=TIME_STEPS,
    sampling_timesteps=SAMPLING_TIMESTEPS
)

model = model.to(device)
diffusion = diffusion.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

In [ ]:
images, labels = next(iter(loader))
images = images.to(device)

loss = diffusion(images)

print("Image batch shape:", images.shape)
print("Loss:", loss.item())

In [ ]:
# training loop
train_losses = []

for epoch in range(N_EPOCHS):
    model.train()
    epoch_losses = []

    start_time = time.time()

    progress_bar = tqdm(loader, desc=f"Epoch {epoch + 1}/{N_EPOCHS}")

    for images, labels in progress_bar:
        images = images.to(device)

        loss = diffusion(images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())
        progress_bar.set_postfix(loss=loss.item())

    epoch_loss = np.mean(epoch_losses)
    train_losses.append(epoch_loss)

    elapsed = time.time() - start_time
    print(f"Epoch {epoch + 1}: loss = {epoch_loss:.4f}, time = {elapsed:.1f} s")

    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(), f"../models/A7_mnist_diffusion_epoch_{epoch + 1}.pth")

In [ ]:
# plot training loss
plt.figure(figsize=(6, 4))
plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("DDPM training loss on MNIST")
plt.grid(True)
plt.savefig("../plots/A7_training_loss.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# generate samples
model.eval()

with torch.no_grad():
    sampled_images = diffusion.sample(batch_size=16)

sampled_images = sampled_images.cpu()
sampled_images = sampled_images.clamp(0, 1)

grid = torchvision.utils.make_grid(sampled_images, nrow=4)

plt.figure(figsize=(5, 5))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.axis("off")
plt.title("Generated MNIST samples")
plt.savefig("../plots/A7_generated_mnist_samples.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Compare generated images to real images
real_images, real_labels = next(iter(test_loader))
real_images = real_images[:16]

real_grid = torchvision.utils.make_grid(real_images, nrow=4)
sample_grid = torchvision.utils.make_grid(sampled_images[:16], nrow=4)

plt.figure(figsize=(5, 5))
plt.imshow(real_grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.axis("off")
plt.title("Real MNIST examples")
plt.savefig("../plots/A7_real_mnist_grid.png", dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(5, 5))
plt.imshow(sample_grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.axis("off")
plt.title("Generated MNIST examples")
plt.savefig("../plots/A7_generated_mnist_grid.png", dpi=200, bbox_inches="tight")
plt.show()